# 06 -- Линейная модель

Проверяю, достаточно ли линейного сочетания тех же 101 признака. Масштабирование учу только на rank_train.
Все случайные операции используют seed 143.

In [1]:
from pathlib import Path
import sys
import json
import time
import random

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from IPython.display import display

ln_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
import nbformat

# Подключаю только ячейки с функциями; расчеты других ноутбуков не запускаю.
for ln_file in ['02_feature_engineering.ipynb']:
    for ln_cell in nbformat.read(ln_root / 'notebooks' / ln_file, as_version=4).cells:
        if 'definitions' in ln_cell.metadata.get('tags', []):
            exec(ln_cell.source, globals())
ln_out = ln_root / 'outputs'
ln_seed = 143
random.seed(ln_seed)
np.random.seed(ln_seed)

## Обучающие кандидаты

Группы без найденного позитива исключаю только из обучения. На dev они остаются в знаменателе.
Для классификаторов суммарный вес положительной и неразмеченной части каждого запроса одинаковый.
Каталожные ID и target не попадают в матрицу. Масштабирование обучаю только на rank_train.

In [2]:
ln_train = (
    pd.read_parquet(ln_out / 'cache/extended_rank_train.parquet')
    .sort_values(['query_id', 'pos'])
    .reset_index(drop=True)
)
ln_dev = (
    pd.read_parquet(ln_out / 'cache/extended_dev.parquet')
    .sort_values(['query_id', 'pos'])
    .reset_index(drop=True)
)
ln_queries = pd.read_parquet(ln_out / 'cache/eval_queries.parquet')
ln_dev_queries = ln_queries[ln_queries.split.eq('dev')]
ln_items = pd.read_parquet(ln_out / 'cache/items.parquet', columns=['item_id'])
ln_truth = json.loads((ln_out / 'cache/truth.json').read_text())
ln_dev_truth = {q: ln_truth[q] for q in ln_dev_queries.query_id}
ln_ids = ln_items.item_id.to_numpy()
ln_columns = feature_columns(ln_train)
ln_train['target'] = [
    int(ln_ids[p] in ln_truth[q]) for q, p in zip(ln_train.query_id, ln_train.pos)
]
ln_found = ln_train.groupby('query_id').target.transform('sum')
ln_train = ln_train[ln_found.gt(0)].reset_index(drop=True)
ln_npos = ln_train.groupby('query_id').target.transform('sum')
ln_n = ln_train.groupby('query_id').target.transform('size')
ln_weights = np.where(
    ln_train.target.eq(1), 0.5 / ln_npos, 0.5 / (ln_n - ln_npos).clip(lower=1)
)
ln_weights /= ln_weights.mean()
ln_x = model_matrix(ln_train, ln_columns)
ln_xdev = model_matrix(ln_dev, ln_columns)
ln_share = json.loads((ln_out / 'models/hybrid.json').read_text())['warm_share']
ln_results = []
ln_scores = ln_dev[['query_id', 'pos']].copy()
write_json(
    {'features': ln_columns, 'seed': ln_seed}, ln_out / 'models/extended_features.json'
)

In [3]:
def ln_evaluate(name, scores, seconds):
    predictions = predictions_from_frame(
        ln_dev.assign(score=scores), ln_items, score='score', k=100
    )
    per_query = recall_table(predictions, ln_dev_truth).merge(
        ln_dev_queries[['query_id', 'regime']], on='query_id'
    )
    values = per_query.groupby('regime')['recall@50'].mean()
    ln_scores[name] = scores
    ln_results.append(
        {
            'model': name,
            'cold': values['cold'],
            'warm': values['warm'],
            'objective': (1 - ln_share) * values['cold'] + ln_share * values['warm'],
            'fit_seconds': seconds,
        }
    )
    pd.DataFrame(ln_results).to_csv(ln_out / 'validation/models_ln.csv', index=False)
    per_query.to_parquet(ln_out / f'validation/dev_{name}.parquet', index=False)
    print(ln_results[-1], flush=True)

## Обучение

Проверяю, достаточно ли линейно сложить сигналы. SGD позволяет обучить логистическую модель
на всем пуле без большой матрицы вторых производных. Использую те же веса запросов.

In [4]:
ln_scaler = StandardScaler().fit(ln_x)
ln_z = np.clip(ln_scaler.transform(ln_x), -10, 10).astype('float32')
ln_zdev = np.clip(ln_scaler.transform(ln_xdev), -10, 10).astype('float32')
ln_start = time.perf_counter()
ln_linear = SGDClassifier(
    loss='log_loss', alpha=0.001, max_iter=20, tol=1e-4, random_state=ln_seed
)
ln_linear.fit(ln_z, ln_train.target, sample_weight=ln_weights)
ln_evaluate(
    'linear', ln_linear.decision_function(ln_zdev), time.perf_counter() - ln_start
)
joblib.dump({'model': ln_linear, 'scaler': ln_scaler}, ln_out / 'models/linear.joblib')

/Users/dvsorochan/.codex/.chatgpt-projects/g-p-6ab7bad8eeb481919ab7a788d369d736/.venv/lib/python3.12/site-packages/sklearn/linear_model/_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


{'model': 'linear', 'cold': np.float64(0.7878055555555555), 'warm': np.float64(0.8043452380952381), 'objective': np.float64(0.7940045638157383), 'fit_seconds': 9.069299499999943}


['/Users/dvsorochan/.codex/.chatgpt-projects/g-p-6ab7bad8eeb481919ab7a788d369d736/avito_rebuilt/outputs/models/linear.joblib']

In [5]:
ln_scores.to_parquet(ln_out / 'cache/scores_ln.parquet', index=False)
pd.DataFrame(ln_results).to_csv(ln_out / 'validation/models_ln.csv', index=False)
display(pd.DataFrame(ln_results))

,model,cold,warm,objective,fit_seconds
0,linear,0.787806,0.804345,0.794005,9.069299


## Вывод

Логистическая модель дала 0.79400 на dev против 0.80617 у CatBoost с теми же признаками.
Линейного сочетания сигналов здесь оказалось недостаточно. Оставляю этот результат как
контроль необходимости нелинейной модели.